In [1]:
# Memory safety: cap this kernel to the RAM free right now so an out-of-memory
# operation fails with a clean MemoryError instead of crashing VS Code /
# thrashing swap. For very large parquet loads, read_parquet_float32() from this
# module keeps peak memory bounded.
import os, sys
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *
install_memory_guard()


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 10.8G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


14298144768

In [2]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")); from parallel_compute import *

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
import pyarrow as pa
import pyarrow.parquet as pq

# Stream the source in row batches,
# keep only the in-window rows, and write them straight to disk - peak RAM stays
# at ~one batch.
start = variables.FEATURE_SELECTION_START_DATE
end = variables.FEATURE_SELECTION_END_DATE
pf = pq.ParquetFile(variables.FEATURES_DATASET_PATH, pre_buffer=False, memory_map=False)
out_path = variables.FEATURES_DATASET_FOR_SELECTION_PATH
writer = None
n_written = 0
try:
    for rb in pf.iter_batches(batch_size=50_000, use_threads=False):
        df = rb.to_pandas()
        sub = df[(df.index >= start) & (df.index <= end)]
        if len(sub):
            table = pa.Table.from_pandas(sub, preserve_index=True)
            if writer is None:
                writer = pq.ParquetWriter(out_path, table.schema)
            else:
                table = table.cast(writer.schema)
            writer.write_table(table)
            n_written += len(sub)
        del df, sub
finally:
    if writer is not None:
        writer.close()
print("Feature-selection matrix rows:", n_written)

Feature-selection matrix rows: 525673


Free this kernel's memory so the next notebook has RAM to work with


(clears data variables + returns freed heap to the OS).

In [4]:
release_memory()


[release_memory] cleared 14 variable(s); kernel rss 0.47G, 10.7G RAM free now
